In [66]:
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## padding & concatenation

In [67]:
# Load product embeddings
cache = np.load("fashionclip_embeddings.npz")
embedding_dict = {product_id: cache[product_id] for product_id in cache.files}

# Load product categories
with open("product_category.json", "r") as f:
    product_category = json.load(f)

# Load cleaned outfits
df = pd.read_csv("clean_outfits_compatibility_prediction.csv")
df["products"] = df["products"].apply(json.loads)

# Fixed category order
categories = ["OUTERWEAR", "TOP", "BOTTOM", "FOOTWEAR"]
embedding_dim = 512

def get_outfit_embedding(products):

    # Initialize all category slots with zeros
    category_embeddings = {category: np.zeros(embedding_dim) for category in categories}

    # Fill the corresponding category slots
    for pid in products:
        if pid in embedding_dict and pid in product_category:
            category = product_category[pid]
            if category in category_embeddings:
                category_embeddings[category] = embedding_dict[pid]

    # Concatenate embeddings in fixed category order
    return np.concatenate([category_embeddings[category] for category in categories])

df["embedding"] = df["products"].apply(get_outfit_embedding)

print("Outfit embedding dimension:", len(df["embedding"].iloc[0]))
df

Outfit embedding dimension: 2048


,outfit_id,label,products,embedding
0,O0001,1,"[119704139_2, 119704139_1, 119704139_3, 119704...","[-0.34925437, -0.106640786, 0.16303492, 0.4914..."
1,O0002,1,"[119314458_1, 119314458_2]","[-1.0425564050674438, 1.5769972801208496, -0.7..."
2,O0003,1,"[147361785_1, 147361785_2, 147361785_3]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,O0005,1,"[192637078_2, 192637078_1]","[-0.4453507363796234, -0.00700630247592926, -0..."
4,O0006,1,"[163560873_1, 163560873_2, 163560873_3]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...
2372,O7058,0,"[136727316_2, 126616464_1, 107813921_2]","[-0.5429734587669373, -0.027578962966799736, -..."
2373,O7068,0,"[216946108_1, 129744291_3]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2374,O7070,0,"[122847911_2, 103461964_1, 195200115_3, 216731...","[-0.089611545, 0.073378146, 0.17025855, 0.0594..."
2375,O7072,0,"[209222683_1, 184041913_1, 207694450_3]","[-0.040070995688438416, -0.11683124303817749, ..."


## model train

In [68]:
X = np.stack(df["embedding"].values)
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Split temporary data into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1663, 2048)
Validation: (357, 2048)
Test: (357, 2048)


In [69]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(16,),
    activation="relu",
    alpha=0.1,
    max_iter=300,
    early_stopping=True,
    random_state=42
)

mlp.fit(X_train, y_train)

MLPClassifier(alpha=0.1, early_stopping=True, hidden_layer_sizes=(16,),
              max_iter=300, random_state=42)

In [70]:
val_pred = mlp.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print("Validation Classification Report:")
print(classification_report(y_val, val_pred, target_names=["Incompatible", "Compatible"]))

test_pred = mlp.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, test_pred))
print("Test Classification Report:")
print(classification_report(y_test, test_pred, target_names=["Incompatible", "Compatible"]))

Validation Accuracy: 0.5742296918767507
Validation Classification Report:
              precision    recall  f1-score   support

Incompatible       0.54      0.51      0.53       165
  Compatible       0.60      0.63      0.61       192

    accuracy                           0.57       357
   macro avg       0.57      0.57      0.57       357
weighted avg       0.57      0.57      0.57       357


Test Accuracy: 0.6190476190476191
Test Classification Report:
              precision    recall  f1-score   support

Incompatible       0.60      0.55      0.57       166
  Compatible       0.64      0.68      0.65       191

    accuracy                           0.62       357
   macro avg       0.62      0.61      0.61       357
weighted avg       0.62      0.62      0.62       357



In [71]:
pipeline = Pipeline([("scaler", scaler), ("mlp", mlp)])
joblib.dump(pipeline, "outfits_compatibility_model.pkl")

['outfits_compatibility_model.pkl']